# Finding 3 — The Democratization Paradox
**Study:** Mapping UPI: Scale, Geography, and Competition in India's Payment Ecosystem
**Data:** NPCI App-wise Statistics (2020–2026)
**Analysis:** Herfindahl-Hirschman Index (HHI) monthly trend, PhonePe-GPay duopoly analysis, Paytm regulatory collapse, long tail problem

In [ ]:
import subprocess, sys

for pkg in ["matplotlib", "seaborn", "pandas", "numpy", "sqlalchemy", "pymysql"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import seaborn as sns
from sqlalchemy import create_engine
from pathlib import Path

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", font_scale=1.6)
plt.rcParams.update({
    "figure.dpi": 200,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print("All dependencies ready.")

In [ ]:
import re

# MySQL connection — set your own credentials via environment or edit below.
DB_USER = "your_username"    # your MySQL username
DB_PASS = "your_password"    # your MySQL password
DB_HOST = "localhost"        # or your MySQL host
DB_NAME = "upi_data"

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}?charset=utf8mb4"
)

# App-wise volume — NPCI Ecosystem Statistics 2020–2026
df = pd.read_sql("""
    SELECT month_year, app_name, total_volume_mn
    FROM upi_apps
    ORDER BY month_year, app_name
""", engine)

# Drop nulls/zeros, filter to 2020+
df = df.dropna(subset=["total_volume_mn"])
df = df[df["total_volume_mn"] > 0].copy()
df = df[df["month_year"] >= "2020-01"].copy()
df["date"] = pd.to_datetime(df["month_year"] + "-01")

# Strip trailing ' #' to get clean display name
df["app_clean"] = df["app_name"].str.replace(r"\s*#$", "", regex=True).str.strip()

# Canonical name map (NPCI renamed apps over time)
# PhonePe was stored as "PhonePe", "Phonepe", "Phone Pe" across years
CANONICAL = [
    (r"phone\s*pe|phonepe",         "PhonePe"),
    (r"google\s*pay|gpay|tez",      "Google Pay"),
    (r"paytm",                      "Paytm"),
    (r"^bhim$",                     "BHIM"),
    (r"amazon\s*pay",               "Amazon Pay"),
    (r"whatsapp",                   "WhatsApp"),
]

def to_canonical(name):
    n = name.lower().strip()
    for pattern, canonical in CANONICAL:
        if re.search(pattern, n):
            return canonical
    return name

df["app_std"] = df["app_clean"].apply(to_canonical)

# Re-aggregate after standardisation (merges all PhonePe variants, etc.)
df_std = (
    df.groupby(["month_year", "date", "app_std"])["total_volume_mn"]
    .sum()
    .reset_index()
)
monthly_total = df_std.groupby("month_year")["total_volume_mn"].transform("sum")
df_std["market_share"] = df_std["total_volume_mn"] / monthly_total * 100

print(f"Raw rows     : {len(df):,}   unique raw names: {df['app_clean'].nunique()}")
print(f"Merged rows  : {len(df_std):,}  unique std names: {df_std['app_std'].nunique()}")
print(f"Months       : {df_std['month_year'].nunique()}  ({df_std['month_year'].min()} to {df_std['month_year'].max()})")

# Sanity: check shares sum to ~100 each month
check = df_std.groupby("month_year")["market_share"].sum()
print(f"\nShare sum — min: {check.min():.1f}%  max: {check.max():.1f}%  (should be ~100)")

In [ ]:
# HHI from standardised data (correct — PhonePe variants merged)
hhi = (
    df_std.groupby("month_year")["market_share"]
    .apply(lambda s: (s ** 2).sum())
    .reset_index(name="hhi")
)
hhi["date"] = pd.to_datetime(hhi["month_year"] + "-01")

print(hhi.tail(6).to_string(index=False))

## Chart 1 — HHI Trend (2020–2026)

In [ ]:
dates  = hhi["date"].values
values = hhi["hhi"].values
y_top  = max(values.max() * 1.18, 4200)

fig, ax = plt.subplots(figsize=(20, 11))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# Zone bands
ax.axhspan(0,    1500,  color="#bbf7d0", alpha=1.0, zorder=0)
ax.axhspan(1500, 2500,  color="#fef9c3", alpha=1.0, zorder=0)
ax.axhspan(2500, y_top, color="#fee2e2", alpha=1.0, zorder=0)

ax.axhline(1500, color="#16a34a", lw=1.5, ls="--", alpha=0.6, zorder=1)
ax.axhline(2500, color="#dc2626", lw=1.5, ls="--", alpha=0.6, zorder=1)

ax.fill_between(dates, values, alpha=0.12, color="#1e3a5f", zorder=2)
ax.plot(dates, values, color="#1e3a5f", lw=3.2, zorder=3, solid_capstyle="round")
ax.scatter(dates, values, color="#1e3a5f", s=30, zorder=4,
           edgecolors="white", linewidths=0.8)

# Current value callout
ax.scatter([dates[-1]], [values[-1]], color="#1e3a5f", s=160, zorder=5,
           edgecolors="white", linewidths=2.5)
ax.annotate(
    f"Current ({hhi['month_year'].iloc[-1]})\nHHI = {values[-1]:,.0f}",
    xy=(dates[-1], values[-1]),
    xytext=(-120, 35), textcoords="offset points",
    fontsize=14, fontweight="bold", color="#1e3a5f",
    bbox=dict(boxstyle="round,pad=0.5", fc="white", ec="#1e3a5f", alpha=0.95),
    arrowprops=dict(arrowstyle="->", color="#1e3a5f", lw=1.5)
)

# First crossing of 2500 annotation
# Placed bottom-right of the crossing point to stay clear of zone label
over2500 = hhi[hhi["hhi"] >= 2500]
if not over2500.empty:
    first = over2500.iloc[0]
    ax.annotate(
        f"Enters highly\nconcentrated zone\n{first['month_year']}",
        xy=(first["date"], first["hhi"]),
        xytext=(55, -95), textcoords="offset points",
        fontsize=13, color="#991b1b",
        arrowprops=dict(arrowstyle="->", color="#991b1b", lw=1.3,
                        connectionstyle="arc3,rad=0.30"),
        bbox=dict(boxstyle="round,pad=0.45", fc="white", ec="#991b1b", alpha=0.92)
    )

# Zone labels — anchored near the TOP of each band so they
#    never collide with the data line which sits in the red zone
ax.text(0.012, 750,  "Competitive  (< 1,500)",
        transform=ax.get_yaxis_transform(), color="#15803d",
        fontsize=14, fontweight="bold", va="center")
ax.text(0.012, 2000, "Moderate  (1,500 – 2,500)",
        transform=ax.get_yaxis_transform(), color="#a16207",
        fontsize=14, fontweight="bold", va="center")
# "Highly Concentrated" label moved to near top of the red band
ax.text(0.012, y_top * 0.93, "Highly Concentrated  (> 2,500)",
        transform=ax.get_yaxis_transform(), color="#991b1b",
        fontsize=14, fontweight="bold", va="top")

# Axes & styling
ax.set_xlim(dates[0], dates[-1])
ax.set_ylim(0, y_top)
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"{y:,.0f}"))
ax.tick_params(axis="x", rotation=30, labelsize=14)
ax.tick_params(axis="y", labelsize=14)
ax.set_ylabel("HHI Score", fontsize=16)
ax.set_title("UPI Market Concentration — Herfindahl-Hirschman Index (HHI)",
             fontsize=20, fontweight="bold", pad=16)
ax.grid(axis="y", color="white", lw=1.5, zorder=1)
ax.grid(axis="x", alpha=0)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT / "hhi_trend.png")
plt.show()
print("Saved -> output/hhi_trend.png")

## Chart 2 — Top 5 Apps Market Share (Stacked Area)

In [ ]:
# Top 5 by average share across all months (standardised names)
top5 = (
    df_std.groupby("app_std")["market_share"]
    .mean()
    .nlargest(5)
    .index.tolist()
)
print("Top 5 apps (merged names):")
for i, app in enumerate(top5, 1):
    avg = df_std[df_std["app_std"] == app]["market_share"].mean()
    print(f"  {i}. {app:<25} avg {avg:.1f}%")

all_months = sorted(df_std["month_year"].unique())
pivot = (
    df_std[df_std["app_std"].isin(top5)]
    .pivot_table(index="month_year", columns="app_std",
                 values="market_share", aggfunc="sum")
    .reindex(all_months)
    .fillna(0)
)
col_order = (
    df_std[df_std["app_std"].isin(top5)]
    .groupby("app_std")["market_share"].mean()
    .reindex(top5).sort_values(ascending=False).index.tolist()
)
pivot["Others"] = (100 - pivot.sum(axis=1)).clip(lower=0)
pivot.index = pd.to_datetime(pivot.index + "-01")
pivot = pivot[col_order + ["Others"]]

BRAND = {
    "PhonePe":    "#5b21b6",
    "Google Pay": "#1d4ed8",
    "Paytm":      "#0891b2",
    "BHIM":       "#c2410c",
    "Amazon Pay": "#b45309",
    "WhatsApp":   "#15803d",
    "Others":     "#94a3b8",
}
fallback = sns.color_palette("tab10", 10)
colors = [BRAND.get(c, fallback[i % 10]) for i, c in enumerate(pivot.columns)]

fig, ax = plt.subplots(figsize=(20, 12))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

ax.stackplot(
    pivot.index,
    [pivot[col].values for col in pivot.columns],
    labels=pivot.columns.tolist(),
    colors=colors,
    alpha=0.88
)

# Band labels (white text inside each band)
cumsum = pivot.cumsum(axis=1)
mid_x  = pivot.index[len(pivot) // 2]
for j, col in enumerate(pivot.columns):
    y_hi  = cumsum[col].loc[mid_x]
    y_lo  = cumsum.iloc[:, j - 1].loc[mid_x] if j > 0 else 0
    y_mid = (y_hi + y_lo) / 2
    band_h = y_hi - y_lo
    if band_h >= 5:
        ax.text(mid_x, y_mid, col,
                fontsize=14, va="center", ha="center",
                color="white" if col != "Others" else "#334155",
                fontweight="bold")

# Right-edge share labels
for j, col in enumerate(pivot.columns):
    y_hi  = cumsum[col].iloc[-1]
    y_lo  = cumsum.iloc[:, j - 1].iloc[-1] if j > 0 else 0
    share = pivot[col].iloc[-1]
    if share >= 4:
        ax.text(pivot.index[-1] + pd.DateOffset(days=14),
                (y_hi + y_lo) / 2,
                f"{share:.1f}%",
                fontsize=13, va="center", color=colors[j],
                fontweight="bold")

ax.set_xlim(pivot.index.min(), pivot.index.max() + pd.DateOffset(months=2))
ax.set_ylim(0, 100)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=7))
ax.xaxis.set_minor_formatter(mdates.DateFormatter("Jul"))
ax.tick_params(axis="x", which="major", labelsize=14, length=5)
ax.tick_params(axis="x", which="minor", labelsize=12, length=3, colors="#9ca3af")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"{y:.0f}%"))
ax.tick_params(axis="y", labelsize=14)
ax.set_ylabel("Market Share (%)", fontsize=16)
ax.set_title("Top 5 UPI Apps — Monthly Market Share by Transaction Volume",
             fontsize=20, fontweight="bold", pad=16)
ax.grid(axis="y", alpha=0.2, lw=0.8, color="#1e293b")
ax.grid(axis="x", alpha=0)
handles, labels_ = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels_[::-1],
          loc="upper center", bbox_to_anchor=(0.5, -0.08),
          ncol=6, fontsize=13, framealpha=0.95, edgecolor="#e5e7eb")
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT / "top5_market_share.png")
plt.show()
print("Saved -> output/top5_market_share.png")

## Key Statistics

In [ ]:
latest_month = df_std["month_year"].max()
latest_df    = df_std[df_std["month_year"] == latest_month]
current_hhi  = hhi[hhi["month_year"] == latest_month]["hhi"].values[0]

zone = ("Highly Concentrated" if current_hhi > 2500
        else "Moderately Concentrated" if current_hhi > 1500
        else "Competitive")

over2500    = hhi[hhi["hhi"] >= 2500]
first_cross = over2500.iloc[0]["month_year"] if not over2500.empty else "Never"

phonepay_share = latest_df[latest_df["app_std"] == "PhonePe"]["market_share"].sum()
gpay_share     = latest_df[latest_df["app_std"] == "Google Pay"]["market_share"].sum()

print(f"Latest month: {latest_month}")
breakdown = latest_df.groupby("app_std")["market_share"].sum().sort_values(ascending=False).head(10)
print("\nTop 10 apps by share:")
for app, share in breakdown.items():
    print(f"  {app:<30} {share:.1f}%")

print()
print("=" * 52)
print("  KEY FINDINGS")
print("=" * 52)
print(f"  HHI first crossed 2500       : {first_cross}")
print(f"  Current HHI  ({latest_month})      : {current_hhi:,.0f}  [{zone}]")
print(f"  PhonePe share  ({latest_month})  : {phonepay_share:.1f}%")
print(f"  Google Pay share ({latest_month}): {gpay_share:.1f}%")
print(f"  PhonePe + GPay combined      : {phonepay_share + gpay_share:.1f}%")
print("=" * 52)

---

## Chart 3 — PhonePe vs Google Pay: The Duopoly Race
Do two players dominate UPI, and is either one breaking the NPCI 30% cap?

## Chart 4 — Paytm's Regulatory Collapse
How much market share did Paytm lose after the RBI action in January 2024?

In [ ]:
# PhonePe vs Google Pay head-to-head race
duo = (
    df_std[df_std["app_std"].isin(["PhonePe", "Google Pay"])]
    .pivot_table(index="month_year", columns="app_std",
                 values="market_share", aggfunc="sum")
    .fillna(0)
)
duo.index = pd.to_datetime(duo.index + "-01")
duo = duo.sort_index()
duo["Combined"] = duo.sum(axis=1)

fig, ax = plt.subplots(figsize=(20, 11))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# Shaded gap between the two lines
ax.fill_between(duo.index, duo["PhonePe"], duo["Google Pay"],
                where=duo["PhonePe"] >= duo["Google Pay"],
                alpha=0.10, color="#5b21b6")
ax.fill_between(duo.index, duo["PhonePe"], duo["Google Pay"],
                where=duo["PhonePe"] < duo["Google Pay"],
                alpha=0.10, color="#1d4ed8")

ax.plot(duo.index, duo["PhonePe"],    color="#5b21b6", lw=3.0, label="PhonePe")
ax.plot(duo.index, duo["Google Pay"], color="#1d4ed8", lw=3.0, label="Google Pay")
ax.plot(duo.index, duo["Combined"], color="#374151", lw=2.2,
        ls="--", alpha=0.75, label="PhonePe + GPay (combined)")
ax.fill_between(duo.index, duo["Combined"], 100,
                alpha=0.04, color="#374151")

# NPCI 30% cap guideline
ax.axhline(30, color="#dc2626", lw=1.2, ls=":", alpha=0.6)
ax.text(duo.index[1], 31.5, "NPCI 30% market cap guideline",
        fontsize=13, color="#dc2626", style="italic")

# RBI restricts Paytm — Jan 2024: combined share steps up visibly
paytm_event = pd.Timestamp("2024-01-01")
ax.axvline(paytm_event, color="#dc2626", lw=1.5, ls="--", alpha=0.7)
idx = duo.index.get_indexer([paytm_event], method="nearest")[0]
combined_at_event = duo["Combined"].iloc[idx]
ax.annotate(
    "RBI restricts Paytm --\nduopoly absorbs redistributed volume",
    xy=(paytm_event, combined_at_event),
    xytext=(18, 35), textcoords="offset points",
    fontsize=8, color="#dc2626",
    bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="#dc2626", alpha=0.88),
    arrowprops=dict(arrowstyle="->", color="#dc2626", lw=1.1)
)

# End labels
phonePe_val  = duo["PhonePe"].iloc[-1]
gpay_val     = duo["Google Pay"].iloc[-1]
combined_val = duo["Combined"].iloc[-1]

ax.annotate(f"PhonePe: {phonePe_val:.1f}%",
            xy=(duo.index[-1], phonePe_val),
            xytext=(-130, 18), textcoords="offset points",
            fontsize=13, color="#5b21b6", fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#5b21b6", alpha=0.92),
            arrowprops=dict(arrowstyle="->", color="#5b21b6", lw=1.2))

ax.annotate(f"Google Pay: {gpay_val:.1f}%",
            xy=(duo.index[-1], gpay_val),
            xytext=(-145, -30), textcoords="offset points",
            fontsize=13, color="#1d4ed8", fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#1d4ed8", alpha=0.92),
            arrowprops=dict(arrowstyle="->", color="#1d4ed8", lw=1.2))

ax.annotate(f"Combined: {combined_val:.1f}%",
            xy=(duo.index[-1], combined_val),
            xytext=(-145, 22), textcoords="offset points",
            fontsize=13, color="#374151", fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#374151", alpha=0.92),
            arrowprops=dict(arrowstyle="->", color="#374151", lw=1.2))

ax.set_xlim(duo.index.min(), duo.index.max())
ax.set_ylim(0, min(duo["Combined"].max() * 1.18, 100))
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"{y:.0f}%"))
ax.tick_params(axis="x", rotation=30, labelsize=14)
ax.tick_params(axis="y", labelsize=14)
ax.set_ylabel("Market Share (%)", fontsize=16)
ax.set_title("PhonePe vs Google Pay -- Head-to-Head Market Share Race",
             fontsize=20, fontweight="bold", pad=16)
ax.legend(fontsize=13, framealpha=0.95, loc="lower left", edgecolor="#e5e7eb")
ax.grid(axis="y", alpha=0.3, lw=0.6)
ax.grid(axis="x", alpha=0)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT / "duopoly_race.png")
plt.show()
print("Saved -> output/duopoly_race.png")

In [ ]:
# Paytm market share over time with RBI action annotation
paytm = (
    df_std[df_std["app_std"] == "Paytm"]
    .sort_values("date")
    .set_index("date")["market_share"]
)

rbi_date   = pd.Timestamp("2024-01-01")
avg_before = paytm[paytm.index < rbi_date].mean()
avg_after  = paytm[paytm.index >= rbi_date].mean()

fig, ax = plt.subplots(figsize=(20, 11))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

ax.fill_between(paytm.index, paytm, alpha=0.20, color="#0891b2")
ax.plot(paytm.index, paytm, color="#0891b2", lw=3.0)

# RBI directive line — Jan 31, 2024 (market reacted immediately)
ax.axvline(rbi_date, color="#dc2626", lw=2.0, ls="--", alpha=0.75)
ax.annotate(
    "RBI directive (Jan 31, 2024) --\nmarket share begins collapse",
    xy=(rbi_date, paytm.max() * 0.80),
    xytext=(20, 0), textcoords="offset points",
    fontsize=13, color="#dc2626",
    bbox=dict(boxstyle="round,pad=0.45", fc="white", ec="#dc2626", alpha=0.92)
)

# Before / after averages
ax.plot([paytm.index.min(), rbi_date], [avg_before, avg_before],
        color="#0891b2", lw=1.2, ls=":", alpha=0.7)
ax.plot([rbi_date, paytm.index.max()], [avg_after, avg_after],
        color="#dc2626", lw=1.2, ls=":", alpha=0.7)
ax.text(paytm.index[3], avg_before + 0.30,
        f"Avg before: {avg_before:.1f}%", fontsize=13, color="#0369a1", style="italic")
ax.text(rbi_date + pd.DateOffset(months=2), avg_after + 0.30,
        f"Avg after: {avg_after:.1f}%", fontsize=13, color="#dc2626", style="italic")

# Drop magnitude
drop = avg_before - avg_after
ax.annotate(
    f"−{drop:.1f}pp decline",
    xy=(rbi_date + pd.DateOffset(months=6), (avg_before + avg_after) / 2),
    fontsize=14, color="#7f1d1d", fontweight="bold",
    bbox=dict(boxstyle="round,pad=0.4", fc="#fee2e2", ec="#dc2626", alpha=0.85)
)

ax.set_xlim(paytm.index.min(), paytm.index.max())
ax.set_ylim(0, paytm.max() * 1.35)
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"{y:.1f}%"))
ax.tick_params(axis="x", rotation=30, labelsize=14)
ax.tick_params(axis="y", labelsize=14)
ax.set_ylabel("Market Share (%)", fontsize=16)
ax.set_title("Paytm UPI Market Share — Rise and Regulatory Collapse",
             fontsize=20, fontweight="bold", pad=16)
ax.grid(axis="y", alpha=0.3, lw=0.6)
ax.grid(axis="x", alpha=0)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT / "paytm_collapse.png")
plt.show()
print("Saved -> output/paytm_collapse.png")

## Chart 5 — Total UPI Growth Trajectory
Overall market size growth since 2019 with year-on-year rate.

In [ ]:
agg = pd.read_sql("""
    SELECT month_year, volume_mn, value_cr, banks_on_upi
    FROM upi_aggregate
    WHERE month_year >= '2016-04'
    ORDER BY month_year
""", engine)
agg["date"] = pd.to_datetime(agg["month_year"] + "-01")
agg = agg.dropna(subset=["volume_mn"])
agg = agg[agg["volume_mn"] > 0].sort_values("month_year").copy()
agg["yoy_growth"] = agg["volume_mn"].pct_change(12) * 100

covid_date = pd.Timestamp("2020-03-01")

# Chart 1: Monthly Volume Bar Chart (2016–2026)
fig, ax = plt.subplots(figsize=(20, 11))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

ax.bar(agg["date"], agg["volume_mn"], width=25, color="#1d4ed8", alpha=0.80)
ax.set_ylabel("Monthly Volume (Mn Transactions)", fontsize=16)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"{y:,.0f}"))
ax.set_title("UPI — Total Monthly Transaction Volume (2016–2026)",
             fontsize=20, fontweight="bold", pad=16)
ax.grid(axis="y", alpha=0.3, lw=0.6)
ax.grid(axis="x", alpha=0)
ax.tick_params(axis="y", labelsize=14)
ax.spines[["top", "right"]].set_visible(False)

ax.axvline(covid_date, color="#7c3aed", lw=1.8, ls="--", alpha=0.75)
ax.annotate(
    "COVID lockdown\n(Mar 2020)",
    xy=(covid_date, agg["volume_mn"].max() * 0.55),
    xytext=(12, 0), textcoords="offset points",
    fontsize=13, color="#7c3aed", va="center",
    bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#7c3aed", alpha=0.92)
)

ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax.tick_params(axis="x", rotation=30, labelsize=14)
ax.set_xlim(agg["date"].min(), agg["date"].max())

plt.tight_layout()
plt.savefig(OUTPUT / "upi_volume.png", bbox_inches="tight")
plt.show()
print("Saved -> output/upi_volume.png")

In [ ]:
# Chart 2: YoY Growth Rate (Jan 2020 onwards)
yoy_full = agg.dropna(subset=["yoy_growth"]).copy()
yoy = yoy_full[yoy_full["date"] >= pd.Timestamp("2020-01-01")].copy()

covid_date = pd.Timestamp("2020-03-01")

fig, ax = plt.subplots(figsize=(20, 11))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

ax.plot(yoy["date"], yoy["yoy_growth"], color="#16a34a", lw=2.8)
ax.fill_between(yoy["date"], yoy["yoy_growth"], 0,
                where=yoy["yoy_growth"] >= 0, color="#16a34a", alpha=0.15)
ax.fill_between(yoy["date"], yoy["yoy_growth"], 0,
                where=yoy["yoy_growth"] < 0, color="#dc2626", alpha=0.15)
ax.axhline(0, color="#374151", lw=0.8, alpha=0.4)

ax.axvline(covid_date, color="#7c3aed", lw=1.8, ls="--", alpha=0.75)
y_mid = (yoy["yoy_growth"].max() + yoy["yoy_growth"].min()) / 2
ax.annotate(
    "COVID lockdown\n(Mar 2020)",
    xy=(covid_date, y_mid),
    xytext=(12, 0), textcoords="offset points",
    fontsize=13, color="#7c3aed", va="center",
    bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#7c3aed", alpha=0.92)
)

ax.set_ylabel("YoY Growth (%)", fontsize=16)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"{y:.0f}%"))
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax.tick_params(axis="x", rotation=30, labelsize=14)
ax.tick_params(axis="y", labelsize=14)
ax.set_title("UPI — Year-on-Year Growth Rate (2020–2026)",
             fontsize=20, fontweight="bold", pad=16)
ax.grid(axis="y", alpha=0.3, lw=0.6)
ax.grid(axis="x", alpha=0)
ax.spines[["top", "right"]].set_visible(False)
ax.set_xlim(yoy["date"].min(), yoy["date"].max())

fig.text(
    0.5, -0.04,
    "Growth rate shown from January 2020 onwards. Pre-2020 period excluded: UPI transaction volumes "
    "were below 1 Bn per month before October 2019,\nproducing growth rates exceeding 10,000% that "
    "reflect near-zero base values rather than meaningful acceleration. The post-2020 window\ncaptures "
    "the analytically relevant growth trajectory — COVID inflection, recovery, sustained expansion, and maturation.",
    ha="center", va="top", fontsize=12, color="#6b7280", style="italic"
)

plt.tight_layout()
plt.savefig(OUTPUT / "upi_yoy_growth.png", bbox_inches="tight")
plt.show()
print("Saved -> output/upi_yoy_growth.png")

## Chart 6 — State-wise UPI Distribution
Geographic spread of UPI adoption — which states drive the most volume?

In [ ]:
state_data = pd.read_sql("""
    SELECT state, volume_mn, value_cr
    FROM upi_statewise
    WHERE month_year = (SELECT MAX(month_year) FROM upi_statewise)
      AND state NOT LIKE '%%nclassified%%'
      AND state NOT LIKE '%%thers%%'
      AND state NOT LIKE '%%nknown%%'
      AND state NOT LIKE '%%otal%%'
      AND volume_mn IS NOT NULL
      AND volume_mn > 0
    ORDER BY volume_mn DESC
    LIMIT 15
""", engine)

latest_sw = pd.read_sql(
    "SELECT MAX(month_year) AS m FROM upi_statewise", engine
).iloc[0]["m"]

state_data = state_data.dropna(subset=["volume_mn"])

if state_data.empty:
    print(f"No valid statewise rows for {latest_sw} — check DB data.")
else:
    total_vol = state_data["volume_mn"].sum()
    state_data["share"] = state_data["volume_mn"] / total_vol * 100

    fig, ax = plt.subplots(figsize=(18, 11))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    n = len(state_data)
    bar_colors = plt.cm.Blues(np.linspace(0.35, 0.85, n))[::-1]
    bars = ax.barh(
        state_data["state"].iloc[::-1],
        state_data["volume_mn"].iloc[::-1],
        color=bar_colors, alpha=0.92, height=0.68
    )

    for bar, share, vol in zip(bars,
                                state_data["share"].iloc[::-1],
                                state_data["volume_mn"].iloc[::-1]):
        ax.text(bar.get_width() + state_data["volume_mn"].max() * 0.008,
                bar.get_y() + bar.get_height() / 2,
                f"{vol:,.0f} Mn  ({share:.1f}%)",
                va="center", fontsize=13, color="#374151")

    ax.set_xlabel("Transaction Volume (Millions)", fontsize=16)
    ax.set_title(f"Top 15 States by UPI Transaction Volume — {latest_sw}",
                 fontsize=20, fontweight="bold", pad=16)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.set_xlim(0, state_data["volume_mn"].max() * 1.35)
    ax.grid(axis="x", alpha=0.3, lw=0.6)
    ax.grid(axis="y", alpha=0)
    ax.tick_params(labelsize=14)
    ax.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    plt.savefig(OUTPUT / "statewise.png")
    plt.show()
    print("Saved -> output/statewise.png")
    print(state_data[["state", "volume_mn", "share"]].to_string(index=False))

## Chart 7 — Average Transaction Value by App
Which apps are used for high-value payments vs everyday micro-transactions?

In [ ]:
# Average transaction value per app: (value_cr / volume_mn) * 10 = avg value in ₹
latest_month = df_std["month_year"].max()
app_val = pd.read_sql(f"""
    SELECT app_name, total_volume_mn, total_value_cr
    FROM upi_apps
    WHERE month_year = '{latest_month}'
      AND total_volume_mn > 0.1 AND total_value_cr > 0
""", engine)

app_val["app_std"] = (
    app_val["app_name"]
    .str.replace(r"\s*#$", "", regex=True).str.strip()
    .apply(to_canonical)
)
app_agg = (
    app_val.groupby("app_std")[["total_volume_mn", "total_value_cr"]]
    .sum().reset_index()
)
app_agg["avg_txn_rs"] = app_agg["total_value_cr"] / app_agg["total_volume_mn"] * 10

monthly_share = (
    df_std[df_std["month_year"] == latest_month]
    .groupby("app_std")["market_share"].sum()
)
app_agg = app_agg.merge(monthly_share.rename("share"), on="app_std", how="left")
app_agg = (
    app_agg[app_agg["share"] >= 0.3]
    .sort_values("avg_txn_rs", ascending=False)
    .head(15)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(18, 11))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

bar_colors = ["#1e3a5f" if r > 5000 else "#1d4ed8" if r > 2000 else "#60a5fa"
              for r in app_agg["avg_txn_rs"]]
bars = ax.barh(
    app_agg["app_std"].iloc[::-1],
    app_agg["avg_txn_rs"].iloc[::-1],
    color=bar_colors[::-1], alpha=0.88, height=0.65
)
for bar, val in zip(bars, app_agg["avg_txn_rs"].iloc[::-1]):
    ax.text(bar.get_width() + app_agg["avg_txn_rs"].max() * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"₹{val:,.0f}", va="center", fontsize=13, color="#374151", fontweight="bold")

ax.set_xlabel("Average Transaction Value (₹)", fontsize=16)
ax.set_title(f"Average UPI Transaction Value by App — {latest_month}",
             fontsize=20, fontweight="bold", pad=16)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"₹{x:,.0f}"))
ax.set_xlim(0, app_agg["avg_txn_rs"].max() * 1.28)
ax.grid(axis="x", alpha=0.3, lw=0.6)
ax.grid(axis="y", alpha=0)
ax.tick_params(labelsize=14)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT / "avg_txn_value.png")
plt.show()
print("Saved -> output/avg_txn_value.png")

## Chart 8 — The Long Tail
Many apps hold < 1% market share each. How many are there, and do they matter collectively?

In [ ]:
# Long tail: apps with < 1% market share per month
tail = (
    df_std[df_std["market_share"] < 1]
    .groupby("month_year")
    .agg(n_apps=("app_std", "count"), combined_share=("market_share", "sum"))
    .reset_index()
)
tail["date"] = pd.to_datetime(tail["month_year"] + "-01")
tail = tail.sort_values("date")

avg_share = tail["combined_share"].mean()
peak_n    = tail["n_apps"].max()
latest    = tail.iloc[-1]

fig, ax1 = plt.subplots(figsize=(20, 11))
fig.patch.set_facecolor("white")
ax1.set_facecolor("white")
ax2 = ax1.twinx()

# Bars: number of long-tail apps
ax1.bar(tail["date"], tail["n_apps"], width=25,
        color="#6366f1", alpha=0.78, zorder=2)
ax1.set_ylabel("Number of Apps with < 1% Share", fontsize=16, color="#4338ca")
ax1.tick_params(axis="y", labelcolor="#4338ca", labelsize=14)
ax1.set_ylim(0, tail["n_apps"].max() * 1.22)

# Line: combined share of long tail
ax2.plot(tail["date"], tail["combined_share"],
         color="#dc2626", lw=3.0, zorder=5)
ax2.fill_between(tail["date"], tail["combined_share"],
                 alpha=0.12, color="#dc2626", zorder=3)
ax2.set_ylabel("Combined Market Share (%)", fontsize=16, color="#dc2626")
ax2.tick_params(axis="y", labelcolor="#dc2626", labelsize=14)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"{y:.1f}%"))
ax2.set_ylim(0, tail["combined_share"].max() * 1.35)

ax1.grid(axis="y", alpha=0.25, lw=0.6, zorder=0)
ax1.grid(axis="x", alpha=0)
ax2.grid(False)

# Annotations─
ax1.annotate(
    f"{int(latest['n_apps'])} apps\n< 1% each",
    xy=(tail["date"].iloc[-1], latest["n_apps"]),
    xytext=(-65, 16), textcoords="offset points",
    fontsize=13, color="#4338ca", fontweight="bold",
    bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#4338ca", alpha=0.92),
    arrowprops=dict(arrowstyle="->", color="#4338ca", lw=1.3)
)
ax2.annotate(
    f"{latest['combined_share']:.1f}%\ncombined",
    xy=(tail["date"].iloc[-1], latest["combined_share"]),
    xytext=(-65, -44), textcoords="offset points",
    fontsize=13, color="#dc2626", fontweight="bold",
    bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#dc2626", alpha=0.92),
    arrowprops=dict(arrowstyle="->", color="#dc2626", lw=1.3)
)

# Single centered title — no set_title / suptitle
plt.title(
    "The Long Tail of UPI Apps -- Many Players, Minimal Collective Impact",
    fontsize=20, fontweight="bold", pad=20, loc="center"
)

bar_patch  = mpatches.Patch(color="#6366f1", alpha=0.78, label="# apps with < 1% share")
line_proxy = plt.Line2D([0], [0], color="#dc2626", lw=2.8, label="Combined share of long tail")
ax1.legend(handles=[bar_patch, line_proxy], fontsize=13,
           loc="lower left", framealpha=0.95, edgecolor="#e5e7eb")

ax1.set_xlim(tail["date"].min(), tail["date"].max())
ax1.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax1.tick_params(axis="x", rotation=30, labelsize=14)
ax1.spines[["top"]].set_visible(False)
ax2.spines[["top"]].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT / "long_tail.png")
plt.show()
print("Saved -> output/long_tail.png")